In [18]:
import torch
import torch.nn as nn
from torch.nn import functional as F

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:    device = torch.device("cpu")
print(f'Using device: {device}')

Using device: mps


In [1]:
with open('data/shakespeare.txt', 'r') as f:
    text = f.read()

In [2]:
print(f'Length of dataset in characters: {len(text)}')

Length of dataset in characters: 1115393


In [4]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(f'Vocabulary size: {vocab_size}')


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocabulary size: 65


In [6]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
print(encode('hii there'))
print(decode(encode('hii there')))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [8]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)

torch.Size([1115393]) torch.int64


In [9]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]
print(train_data.shape, val_data.shape)

torch.Size([1003853]) torch.Size([111540])


In [10]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [11]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f'When input is {context} the target: {target}')

When input is tensor([18]) the target: 47
When input is tensor([18, 47]) the target: 56
When input is tensor([18, 47, 56]) the target: 57
When input is tensor([18, 47, 56, 57]) the target: 58
When input is tensor([18, 47, 56, 57, 58]) the target: 1
When input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
When input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
When input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [13]:
torch.manual_seed(1337)
batch_size = 4 # Independent sequences for the batch
block_size = 8 # Context length

def get_batch(split):
    data = train_data if split == 'train' else val_data
    # - block_size because we're gonna take the next 8 characters starting from the index
    # 4 places so batch_size is input
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb)
print('targets:')
print(yb)

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f'When input is {context.tolist()} the target: {target.item()}')

inputs:
tensor([[53, 59,  6,  1, 58, 56, 47, 40],
        [49, 43, 43, 54,  1, 47, 58,  1],
        [13, 52, 45, 43, 50, 53,  8,  0],
        [ 1, 39,  1, 46, 53, 59, 57, 43]])
targets:
tensor([[59,  6,  1, 58, 56, 47, 40, 59],
        [43, 43, 54,  1, 47, 58,  1, 58],
        [52, 45, 43, 50, 53,  8,  0, 26],
        [39,  1, 46, 53, 59, 57, 43,  0]])
When input is [53] the target: 59
When input is [53, 59] the target: 6
When input is [53, 59, 6] the target: 1
When input is [53, 59, 6, 1] the target: 58
When input is [53, 59, 6, 1, 58] the target: 56
When input is [53, 59, 6, 1, 58, 56] the target: 47
When input is [53, 59, 6, 1, 58, 56, 47] the target: 40
When input is [53, 59, 6, 1, 58, 56, 47, 40] the target: 59
When input is [49] the target: 43
When input is [49, 43] the target: 43
When input is [49, 43, 43] the target: 54
When input is [49, 43, 43, 54] the target: 1
When input is [49, 43, 43, 54, 1] the target: 47
When input is [49, 43, 43, 54, 1, 47] the target: 58
When input is

In [19]:
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Tokens read off look up table for next token
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx, targets=None):
        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) #2 dimension array
            targets = targets.view(B*T)

            # Cross entropy expects a B,C,T tensor
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # Focus only on the last time step
            logits = logits[:, -1, :] # becomes (B,C)
            # Apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B,C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B,1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # Append to the running sequence
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(loss)
# Expecting loss to be -ln(1/65) = 4.17

tensor(4.8948, grad_fn=<NllLossBackward0>)


In [20]:
torch.manual_seed(1337)
B, T, C = 4,8,2
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
# version 1 
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t, C)
        xbow[b, t] = torch.mean(xprev, 0)
        # Calculate mean across the batch dimension


In [24]:
x[0], xbow[0]

(tensor([[ 0.1808, -0.0700],
         [-0.3596, -0.9152],
         [ 0.6258,  0.0255],
         [ 0.9545,  0.0643],
         [ 0.3612,  1.1679],
         [-1.3499, -0.5102],
         [ 0.2360, -0.2398],
         [-0.9211,  1.5433]]),
 tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]))

In [ ]:
# version 2
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (T,T) @ (B,T,C)
# PyTorch create a batch dimension --> (B, T, T) @ (B, T, C)
# @ is a batch matrix multiply, apply matrix mul to all batch elements in parallel individually
# Each batch element is T, T and T, C
# (B, T, T) @ (B, T, C) --> (B, T, C)
torch.allclose(xbow, xbow2)
# Batch bow 2 and 1 are equal



True

In [ ]:
# version 3: Softmax
tril = torch.tril(torch.ones(T, T))
# wei is tellign us how much each element from the past should contribute to the current token
wei = torch.zeros((T, T))
# For all elements trill == 0, set the corresponding element in wei to -inf
# Tokens from the past cannot communicate
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

In [33]:
torch.manual_seed(42)
# Make a into a lower triangular matrix
a = torch.tril(torch.ones(3,3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0, 10, (3,2)).float()
c = a @ b
print(a)
print(b)
print(c)
# With this, you start to do sums across rows
# tensor([[1., 0., 0.],
#         [1., 1., 0.],
#         [1., 1., 1.]])
# tensor([[2., 7.],
#         [6., 4.],
#         [6., 5.]])
# tensor([[ 2.,  7.],
#         [ 8., 11.],
#         [14., 16.]])


tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [ ]:
# Version 4
torch.manual_seed(1337)
B, T, C = 4,8,32
x = torch.randn(B, T, C)

head_size = 16
# key embeds the token embedding into a head_size key vector
# the same for query and value
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x) # (B,T,16)
q = query(x) # (B,T,16)
# Tranpose the last 2 dimensions of k to get (B,16,T)
wei = q @ k.transpose(-2, -1) # (B,T,16) @ (B,16,T) --> (B,T,T)

# wei is the raw affinity between the nodes

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
v = value(x)
out = wei @ v

out.shape

# x is private information to a token

torch.Size([4, 8, 16])